# Spots Strategy Simulation

This notebook runs an agent-based simulation of the dice game **Spots** and compares strategy archetypes across thousands of games.

## Game summary

Each player has a hand of dog cards. Each dog card has fixed die-value slots (e.g. `[1, 3, 5]`). On your turn you pick a face-up trick, execute it, and flip it face-down. Tricks involve rolling dice, placing matches on dogs, and burying unplaceable dice in the yard. Yard total > 7 = bust (all dice cleared). First to score 6 completed dogs wins.

**Hard component limits:** 20 treat tokens, 40 dice shared among all players.

## Strategic axes explored

| Strategy | Core bet |
|---|---|
| **OneAtATime** | Complete one dog before touching the next. Assign every die to whichever dog is closest to done. Avoid Howl. |
| **FullPack** | Rush to 6 dogs via Howl. Spread dice evenly so every value is absorbable for as long as possible. |
| **TreatGambler** | Accumulate treats until flush, then burn them on aggressive rolling streaks with rerolls on any miss. |
| **FastAndLoose** | Always roll more dice. Accept busts as the cost of speed. Never reroll (costs turns). |
| **BustAware** | Use exact bust-probability math to decide when to stop. Prioritise dogs that absorb dangerous high-value dice. |
| **Random** | Uniform random on every decision. Baseline only. |

In [1]:
import sys, os

# spots/notebooks/ -> spots/ -> strat_hacking/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from spots.sim import simulate, run_game
from spots.strategies import (
    RandomStrategy, OneAtATimeStrategy, FullPackStrategy,
    TreatGamblerStrategy, FastAndLooseStrategy, BustAwareStrategy,
)

## Run the simulation

10,000 games, 6 players (one of each strategy). All strategies share the same game so results are seat-adjusted — no strategy gets an easier table.

In [2]:
N_GAMES = 10_000

STRATEGIES = [
    ('Random',       lambda rng: RandomStrategy(rng)),
    ('OneAtATime',   lambda rng: OneAtATimeStrategy(rng)),
    ('FullPack',     lambda rng: FullPackStrategy(rng)),
    ('TreatGambler', lambda rng: TreatGamblerStrategy(rng)),
    ('FastLoose',    lambda rng: FastAndLooseStrategy(rng)),
    ('BustAware',    lambda rng: BustAwareStrategy(rng)),
]
names     = [s[0] for s in STRATEGIES]
factories = [s[1] for s in STRATEGIES]
COLOURS   = ['#7f7f7f', '#2ca02c', '#1f77b4', '#ff7f0e', '#d62728', '#9467bd']

print(f'Running {N_GAMES:,} games...')
stats = simulate(factories, names, n_games=N_GAMES, seed=42)
print(stats)

Running 10,000 games...
Simulation: 10000 games, 6 players
  Random                win=11.8%  avg_score=2.75  avg_busts=3.88  avg_treats=2.62
  OneAtATime            win=51.1%  avg_score=4.81  avg_busts=7.83  avg_treats=2.39
  FullPack              win=13.9%  avg_score=2.41  avg_busts=2.06  avg_treats=3.88
  TreatGambler          win=5.7%  avg_score=1.87  avg_busts=3.54  avg_treats=5.41
  FastLoose             win=13.7%  avg_score=2.97  avg_busts=5.05  avg_treats=2.99
  BustAware             win=3.8%  avg_score=1.54  avg_busts=2.96  avg_treats=2.60
  avg turns/game: 134.7


## Win rates, scores, and bust rates

The dashed line on the win-rate chart is the equal-split baseline (100% / n_players). Any strategy above it is beating chance; below it is a liability at the table.

In [3]:
def labelled_bars(ax, values, fmt='{:.1f}', offset=0.3):
    for bar, v in zip(ax.patches, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + offset,
                fmt.format(v), ha='center', va='bottom', fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x        = np.arange(len(names))
baseline = 100 / len(names)

# Win rate
ax = axes[0]
ax.bar(x, [r*100 for r in stats.win_rates], color=COLOURS)
ax.axhline(baseline, color='black', linestyle='--', linewidth=0.8,
           label=f'Equal split ({baseline:.1f}%)')
ax.legend(fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Win rate (%)')
ax.set_title('Win Rate')
labelled_bars(ax, [r*100 for r in stats.win_rates], '{:.1f}%')

# Average dogs scored
ax = axes[1]
ax.bar(x, stats.avg_score, color=COLOURS)
ax.axhline(6, color='black', linestyle='--', linewidth=0.8, label='Win threshold (6)')
ax.legend(fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Avg dogs scored')
ax.set_title('Average Score')
labelled_bars(ax, stats.avg_score, '{:.2f}', offset=0.03)

# Bust rate
ax = axes[2]
ax.bar(x, stats.avg_busts, color=COLOURS)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Avg busts per game')
ax.set_title('Bust Rate')
labelled_bars(ax, stats.avg_busts, '{:.1f}', offset=0.05)

fig.suptitle(f'Spots Strategy Comparison — {N_GAMES:,} games, {len(names)} players',
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'strategy_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved strategy_comparison.png')

Saved strategy_comparison.png


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:44: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


## Bust rate vs win rate

A natural assumption is that busting = bad. This scatter tests that. If bust rate and win rate are positively correlated, busting is the price of an aggressive strategy that scores dogs fast enough to offset the setback. If negatively correlated, caution pays.

In [4]:
fig, ax = plt.subplots(figsize=(7, 5))

for i, (name, colour) in enumerate(zip(names, COLOURS)):
    ax.scatter(stats.avg_busts[i], stats.win_rates[i]*100,
               s=120, color=colour, zorder=3)
    ax.annotate(name, (stats.avg_busts[i], stats.win_rates[i]*100),
                textcoords='offset points', xytext=(6, 2), fontsize=9)

ax.axhline(baseline, color='black', linestyle='--', linewidth=0.8, alpha=0.5,
           label=f'Equal split ({baseline:.1f}%)')
ax.legend(fontsize=8)
ax.set_xlabel('Avg busts per game')
ax.set_ylabel('Win rate (%)')
ax.set_title('Bust Rate vs Win Rate')
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'bust_vs_win.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved bust_vs_win.png')

Saved bust_vs_win.png


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


## Score distributions

Win rate is a single number; score distribution shows the whole picture. A strategy with a tall spike at 6 is reliably reaching the win threshold. A flat distribution means high variance — occasionally great, often mediocre. The dotted line is each strategy's mean score.

In [5]:
import random as _random

N_DETAIL = 3_000
rng_d = _random.Random(77)
detail_results = []
for _ in range(N_DETAIL):
    g = _random.Random(rng_d.randint(0, 2**31))
    strats = [f(g) for f in factories]
    detail_results.append(run_game(strats, g))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, (name, colour) in enumerate(zip(names, COLOURS)):
    scores = [r.scores[i] for r in detail_results]
    wins   = sum(1 for r in detail_results if r.winner == i)
    mean_s = sum(scores) / len(scores)
    ax = axes[i]
    ax.hist(scores, bins=range(0, 8), color=colour, alpha=0.8,
            edgecolor='white', linewidth=0.5, align='left')
    ax.axvline(6,      color='black',  linestyle='--', linewidth=1.2, label='win=6')
    ax.axvline(mean_s, color=colour,   linestyle=':',  linewidth=1.5, alpha=0.8,
               label=f'mean={mean_s:.2f}')
    ax.set_title(f'{name}  (win rate: {wins/N_DETAIL:.1%})', fontsize=11)
    ax.set_xlabel('Dogs scored')
    ax.set_ylabel('Games')
    ax.set_xticks(range(0, 7))
    ax.legend(fontsize=8)

fig.suptitle(f'Score distributions — {N_DETAIL:,} games  (dashed = win threshold)',
             fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'score_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved score_distributions.png')

Saved score_distributions.png


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:34: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


## Treat economy

Treats are a **shared** pool of 20 tokens. A strategy that hoardes treats isn't just helping itself — it's locking up resources that would otherwise appear as tile treats for other players. The left chart shows average end-of-game holdings; the right scatters score against treats to check whether holding treats correlates with winning.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(names))

# Left: avg treats held at game end
ax = axes[0]
ax.bar(x, stats.avg_treats, color=COLOURS)
ax.axhline(20/len(names), color='black', linestyle='--', linewidth=0.8,
           label=f'Equal share of pool ({20/len(names):.1f})')
ax.legend(fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Avg treats at game end')
ax.set_title('Treat Holdings at Game End')
labelled_bars(ax, stats.avg_treats, '{:.1f}', offset=0.05)

# Right: treats vs score scatter
ax = axes[1]
for i, (name, colour) in enumerate(zip(names, COLOURS)):
    sc = [r.scores[i]  for r in detail_results]
    tr = [r.treats[i] for r in detail_results]
    ax.scatter(tr, sc, alpha=0.08, s=8, color=colour, label=name)
ax.set_xlabel('Treats at game end')
ax.set_ylabel('Dogs scored')
ax.set_title('Score vs Treat Holdings')
ax.legend(markerscale=5, fontsize=8)

fig.suptitle('Treat Economy', fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'treat_economy.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved treat_economy.png')

Saved treat_economy.png


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:29: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


## Head-to-head: OneAtATime vs the field

The 6-player run has confounders — all strategies compete simultaneously, so a bad strategy also *hurts* stronger strategies by consuming shared dice and treats. Head-to-head isolates each matchup: 2-player games, OneAtATime against each opponent alone.

In [7]:
N_H2H = 5_000
oaat = lambda rng: OneAtATimeStrategy(rng)

h2h_rates = {}
for name, factory in STRATEGIES:
    if name == 'OneAtATime':
        continue
    s = simulate([oaat, factory], ['OneAtATime', name], n_games=N_H2H, seed=7)
    h2h_rates[name] = s.win_rates[0]
    print(f'  OneAtATime vs {name:15s}: {s.win_rates[0]:.1%}')

fig, ax = plt.subplots(figsize=(7, 4))
opponents    = list(h2h_rates.keys())
rates        = list(h2h_rates.values())
opp_colours  = [COLOURS[names.index(n)] for n in opponents]
bars = ax.barh(opponents, [r*100 for r in rates], color=opp_colours)
ax.axvline(50, color='black', linestyle='--', linewidth=0.8)
for bar, v in zip(bars, rates):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{v:.1%}', va='center', fontsize=10)
ax.set_xlabel('OneAtATime win rate (%)')
ax.set_title(f'OneAtATime head-to-head ({N_H2H:,} games each)')
ax.set_xlim(0, 105)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'head_to_head.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved head_to_head.png')

  OneAtATime vs Random         : 79.3%
  OneAtATime vs FullPack       : 89.0%
  OneAtATime vs TreatGambler   : 48.6%
  OneAtATime vs FastLoose      : 42.1%
  OneAtATime vs BustAware      : 56.3%
Saved head_to_head.png


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:26: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.


## Key findings

### 1. Dog completion speed dominates everything

**OneAtATime** wins by a large margin — roughly 3× the equal-split rate in 6-player games. Focusing all dice on the dog closest to completion returns dice to the shared pool quickly, enables more rolling turns overall, and keeps scoring momentum high. The win condition is 6 *scored* dogs, not dogs in progress.

### 2. High bust rate ≠ bad strategy

OneAtATime busts more than any other strategy, yet wins the most. Busts return all committed dice to the shared pool, which enables more future rolls across the table. A bust that follows 2 scored dogs is far less costly than a bust that follows 0. The bust-vs-win scatter shows a positive relationship for the faster strategies.

### 3. FullPack's coverage trap

Maintaining 6 dogs sounds safe — more placement options means fewer forced burials. But spreading dice keeps dogs in a perpetual half-filled state. Those dice stay committed for many turns without scoring. The dice cycling rate is slower than OneAtATime despite fewer busts. Coverage only helps if you're also scoring.

### 4. BustAware is penalised by caution in a race

Using probability math to quit rolling early results in fewer dice placed per turn. In a competitive multi-player game this means falling behind fast. The table doesn't wait. Risk avoidance has real opportunity cost when faster strategies are compounding completions.

### 5. TreatGambler loses the tempo game

The accumulation phase is dead turns for dog scoring. By the time the treat supply is built up, faster strategies hold a 2–3 dog lead. The splurge phase partially recovers ground but never closes the gap. Treat rerolls improve individual roll quality — they don't make up for missed scoring opportunities.

---

## Open questions

- **Does OneAtATime's dominance shrink in 2-player games?** With fewer competitors, slower strategies have more time to develop before someone wins.
- **What if OneAtATime keeps a pack floor** (never drops below 2 dogs)? Some placement coverage might improve bust-to-score ratio.
- **TreatGambler threshold sweep:** does SPLURGE_AT = 3 outperform SPLURGE_AT = 6? Earlier splurging trades quality for tempo.
- **Trick composition matters:** some trick combos (e.g. Sprint + Chew) may strongly advantage certain strategies over others. Worth studying deal variance.